# VanaDrishti on Colab — 02: Phase 2 benchmark

Thin driver: every cell calls code that lives in the repo (`src/`, `scripts/`), so the notebook stays in sync with the tested code. Outputs go to `MyDrive/vanadrishti/`.

## 1. Setup (run every session)

Before running: **Runtime → Change runtime type → T4 GPU** (any GPU works).
Colab forgets everything except Google Drive when the session ends, so the code and the dataset are
re-fetched each session and all run outputs (checkpoints, logs, metrics) are written to Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/vana-drishti'
BRANCH = 'claude/new-session-uto7x0'
DRIVE = '/content/drive/MyDrive/vanadrishti'
RUNS = f'{DRIVE}/runs'
STATS = f'{DRIVE}/stats/vanadrishti_close_train_stats.pt'
!mkdir -p {RUNS} {DRIVE}/stats {DRIVE}/results

In [ ]:
import os
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q -b {BRANCH} https://github.com/krish-k1301/vana-drishti.git {REPO}
%cd {REPO}
!git log --oneline -1

Pinned versions from `environment.yml`. torch 2.5.1 is required: the upstream reference code calls
`torch.load` in a way that breaks on torch >= 2.6. pip may print dependency-conflict warnings about
other Colab packages; they don't affect this project. All training runs as `!python ...` subprocesses,
so no runtime restart is needed after the install.

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q lightning==2.4.0 torchmetrics==1.9.0 numpy==2.1.3 pandas==2.2.3 scipy==1.17.1 pyyaml==6.0.3 einops==0.8.2 matplotlib==3.11.2 tqdm==4.70.1 requests==2.34.2 pytest==9.1.1
!python -c "import torch, lightning; print(torch.__version__, lightning.__version__, 'CUDA:', torch.cuda.is_available())"

## 2. Dataset (downloaded inside Colab)

BraDD-S1TS from Zenodo: 17.7 GB zip, md5-checked, unzipped to Colab's local disk (fast; Drive is too
small and too slow for 25,988 sample files). The zip is deleted after unzipping to save space.
Takes roughly 10–30 min per session depending on Zenodo's speed; skipped if the data is already there.

In [ ]:
import glob
def find_root():
    hits = sorted(glob.glob(f'{REPO}/data/**/meta.csv', recursive=True))
    return os.path.dirname(hits[0]) if hits else None

if find_root() is None:
    !df -h /content | tail -1
    !python scripts/download_bradd.py --config configs/data/downloads.yaml --delete-zip
ROOT = find_root()
print('BraDD root:', ROOT)
assert ROOT is not None, 'meta.csv not found: check the download output above'

## 3. Phase 2: benchmark (PRD Section 6)

Each model trains with the shared recipe (focal α 0.75, γ 1) and is evaluated on the test split.
U-TAE's reference row is the Phase 1 `ours_focal` run (`baseline_utae_focal`), so run notebook 01
with `RUN = 'ours_focal'` first. Training resumes automatically after a disconnect: rerun the cell.

On a 16 GB T4 everything except AnySat should fit at batch 4; `exchanger_unet` is the heaviest
(8M parameters) — if it runs out of memory, note it rather than changing its batch size silently.
`anysat` needs a checkpoint you supply (`model.params.checkpoint_path`), so it is not in the list.

In [ ]:
MODELS = ['utae_seq2seq', 'convlstm', 'convgru', 'unet3d', 'tsvit', 'galileo', 'exchanger_unet']
MODEL = 'utae_seq2seq'

Only for `exchanger_unet`: its upstream code needs four extra packages; two build from source
(~10 min). Skip this cell for every other model.

In [ ]:
!pip install -q timm==1.0.30 opencv-python-headless==5.0.0.93 setuptools
!pip install -q torch-scatter==2.1.2
!pip install -q --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git@1e3e13bbf607"

In [ ]:
!python -m src.train --config configs/models/{MODEL}.yaml data.root={ROOT} data.stats_path={STATS} data.num_workers=2 output.runs_dir={RUNS}

### Evaluate (pixel/patch metrics with and without the OR rule, size bins, edge vs interior)

In [ ]:
def best_ckpt(exp):
    hits = sorted(glob.glob(f'{RUNS}/{exp}/checkpoints/best*.ckpt'))
    assert hits, f'no best checkpoint for {exp}'
    return hits[-1]

def evaluate(exp, extra=''):
    run = f'{RUNS}/{exp}'
    cmd = f'python -m src.evaluate --config {run}/config_resolved.yaml --checkpoint {best_ckpt(exp)} --split test {extra}'
    print(cmd)
    get_ipython().system(cmd)

evaluate(MODEL)

### U-TAE temporal-depth ablation (PRD Phase 2: 1, 2, 5, 10, all dates)

Uses the Phase 1 `baseline_utae_focal` checkpoint. The 1-date case keeps only the **last** date,
which upstream's `TemporalDropout` cannot do.

In [ ]:
evaluate('baseline_utae_focal')
evaluate('baseline_utae_focal', '--temporal-subsample mode=last_only num_dates=1')
for n in (2, 5, 10):
    evaluate('baseline_utae_focal', f'--temporal-subsample mode=uniform num_dates={n}')

### Benchmark table and plot (U-TAE is the reference row)

In [ ]:
done = [f'{RUNS}/{e}' for e in ['baseline_utae_focal'] + MODELS if glob.glob(f'{RUNS}/{e}/eval_test/metrics.json')]
print('\n'.join(done))
runs_arg = ' '.join(done)
!python -m src.benchmark_table --runs {runs_arg} --out {DRIVE}/results/benchmark.csv
!python scripts/plot_benchmark.py --table {DRIVE}/results/benchmark.csv --out {DRIVE}/results/benchmark.png
import pandas as pd
pd.read_csv(f'{DRIVE}/results/benchmark.csv')